# Mutual Fund Ingestion and Validation

This notebook loads the 10 mutual fund datasets, fetches live NAV histories, explores the fund master structure, and performs data quality checks and AMFI code validations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path
import requests

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Setup data directory
raw_dir = Path("data/raw")
if not raw_dir.exists():
    raw_dir = Path("../data/raw")
print(f"Data directory: {raw_dir.resolve()}")

## 1. Load Datasets and Print Properties

Here we load all the CSV files present in the raw data directory and print their shape, dtypes, and the first few rows.

In [ ]:
csv_files = sorted(list(raw_dir.glob("*.csv")))
datasets = {}
for file_path in csv_files:
    df = pd.read_csv(file_path)
    name = file_path.stem
    datasets[name] = df
    print(f"Dataset: {file_path.name} | Shape: {df.shape}")
    print("Data Types:")
    print(df.dtypes)
    print("Head:")
    display(df.head(3))
    print("=" * 60)

## 2. Detect & Detail Anomalies

We perform checks for:
- Missing values in each dataset
- Duplicate rows in each dataset
- Unexpected value ranges (such as negative values in positive-only columns)

In [ ]:
print("--- Missing Values Check ---")
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"Dataset '{name}' has missing values:")
        for col, count in missing.items():
            print(f"  - Column '{col}': {count} ({count/len(df)*100:.2f}%)")
    else:
        print(f"Dataset '{name}': No missing values")

print("\n--- Duplicate Rows Check ---")
for name, df in datasets.items():
    dups = df.duplicated().sum()
    print(f"Dataset '{name}': {dups} duplicate rows")

print("\n--- Value Range Warnings ---")
for name, df in datasets.items():
    num_cols = df.select_dtypes(include=[np.number]).columns
    for col in num_cols:
        min_val = df[col].min()
        if min_val < 0:
            if 'drawdown' in col or 'return' in col or 'alpha' in col:
                print(f"Dataset '{name}', Column '{col}': has negative values (min: {min_val}) [Expected]")
            else:
                print(f"WARNING: Dataset '{name}', Column '{col}': has negative values (min: {min_val}) [Unexpected]")

## 3. Explore Fund Master & Risk Distribution

We inspect unique categories, sub-categories, risk categories, and plot the distribution of risk categories to visualize the dataset.

In [ ]:
df_fm = datasets.get("01_fund_master")
if df_fm is not None:
    print("Unique Fund Houses in Fund Master:")
    print("  ", df_fm['fund_house'].unique())
    print("\nUnique Categories in Fund Master:")
    print("  ", df_fm['category'].unique())
    print("\nUnique Sub-Categories in Fund Master:")
    print("  ", df_fm['sub_category'].unique())
    print("\nUnique Risk Categories in Fund Master:")
    print("  ", df_fm['risk_category'].unique())
    
    # Plotting
    plt.figure(figsize=(8, 4))
    sns.countplot(data=df_fm, x='risk_category', hue='category', palette='viridis')
    plt.title("Distribution of Schemes by Risk Category & Asset Class")
    plt.xlabel("Risk Category")
    plt.ylabel("Number of Schemes")
    plt.legend(title="Asset Class")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()
else:
    print("Error: '01_fund_master' not found.")

## 4. AMFI Scheme Code Structure & Referential Integrity Validation

Here we analyze:
- How AMFI codes are structured (consecutive pairing between Direct and Regular plans).
- Validate if all scheme codes in `01_fund_master` exist in `nav_history`.

In [ ]:
df_nav_rep = datasets.get("nav_history")
if df_fm is not None and df_nav_rep is not None:
    fm_codes = set(df_fm['amfi_code'].unique())
    nav_codes = set(df_nav_rep['Scheme_Code'].unique())
    
    missing_in_nav = fm_codes - nav_codes
    
    print(f"Unique amfi_codes in 01_fund_master: {len(fm_codes)}")
    print(f"Unique Scheme_Codes in nav_history: {len(nav_codes)}")
    
    if len(missing_in_nav) == 0:
        print("All AMFI scheme codes found.")
    else:
        print("Missing AMFI Codes:")
        for code in sorted(list(missing_in_nav)):
            print(code)
            
    print("\nSample consecutive pairings between Regular and Direct plans:")
    df_plans = df_fm[['amfi_code', 'scheme_name', 'plan', 'fund_house']].sort_values('amfi_code')
    display(df_plans.head(10))
else:
    print("Error: Required datasets for validation not found.")

## 5. Final Summary

### Q&A
- **What unique fields exist in the fund master?** There are 10 unique fund houses, 2 major categories (Equity, Debt), 12 sub-categories, and 5 risk categories/grades ('Moderate', 'Very High', 'Low', 'High', 'Moderately High').
- **Does every AMFI code in `fund_master` exist in `nav_history`?**
  * When comparing `01_fund_master.csv` against `02_nav_history.csv`, they match 1:1 (40/40 codes present).
  * When comparing `01_fund_master.csv` against `nav_history.csv` (originally in reports), 34 AMFI codes are missing since `nav_history.csv` only tracks a subset of 11 schemes.
- **What anomalies exist in the raw datasets?** `04_monthly_sip_inflows` contains 12 missing values in `yoy_growth_pct` (25.00%), which is a mathematical constraint for the first 12 months. No duplicate rows were found in any core dataset. `07_scheme_performance` has negative values in `max_drawdown_pct` (min of -33.5%), which is correct and expected.
- **How are AMFI scheme codes structured?** For newer schemes, Regular and Direct plans are assigned consecutive codes (e.g., SBI Bluechip Regular is 119551 and Direct is 119552). For older schemes, they are widely separated (e.g., HDFC Top 100 Regular is 100016 and Direct is 125497) because Direct plans were introduced in India later (Jan 2013).

### Data Analysis Key Findings
- **Referential Integrity**: Exact matches for core datasets; 34 missing codes when validating `01_fund_master.csv` with `nav_history.csv`.
- **API Mapping Discrepancy**: Standard AMFI codes in the public `mfapi.in` API do not match our local mock codes. For instance, code `125497` (local HDFC Top 100 Direct) fetches SBI Small Cap Direct from the live API. Only Nippon Large Cap (code `118632`) maps to the correct scheme family.
- **Completeness**: 6 newly fetched live NAV CSVs (including HDFC Top 100 `125497`) were successfully saved to `data/raw/` containing up to 3,600 historical NAV records each.

### Insights or Next Steps
- Re-align or map the fetched API NAVs using a custom mapping dictionary (or standard name resolution) when joining with local transaction and portfolio tables.
- Use the newly saved live NAV CSVs for portfolio valuation and tracking performance.